# Model Explainability
Feature importance using built-in XGBoost/Random Forest `feature_importances_` (compatible with all versions).

> **Note:** SHAP's `TreeExplainer` has a known bug with XGBoost >= 2.0 where `base_score` is returned in scientific notation `[5E-1]`. We use native `feature_importances_` instead, which is equally interpretable for HR decision-making.

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

model = joblib.load('../models/v1/attrition_pipeline.joblib')
feature_names = joblib.load('../models/v1/feature_names.joblib')

# Use native feature_importances_ (works for both RandomForest and XGBoost)
importances = model.feature_importances_

feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(15)

print(f'Model type: {type(model).__name__}')
print(f'Total features: {len(feature_names)}')
print()
print('Top 15 Features Driving Attrition Risk:')
print(feat_imp.to_string(index=False))

Model type: XGBClassifier
Total features: 42

Top 15 Features Driving Attrition Risk:
              Feature  Importance
    PerformanceRating    0.869157
OvertimeHoursPerMonth    0.125006
      WorkLifeBalance    0.002141
        TrainingIndex    0.000924
       YearsAtCompany    0.000618
    LastPromotionYear    0.000592
                  Age    0.000521
        IncomePerYear    0.000377
        TrainingHours    0.000373
      ExperienceRatio    0.000290
     JobRole_Helpdesk    0.000000
 Department_Marketing    0.000000
     Department_Sales    0.000000
   Department_Support    0.000000
   JobRole_Accountant    0.000000


In [2]:
# Bar chart of top features
top10 = feat_imp.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#ef4444' if f in ['PerformanceRating', 'WorkLifeBalance', 'OvertimeHoursPerMonth'] else '#38bdf8'
          for f in top10['Feature']]
bars = ax.barh(top10['Feature'][::-1], top10['Importance'][::-1], color=colors[::-1])
ax.set_title('Top 10 Features Driving Attrition Risk\n(Red = Primary HR Action Areas)', fontsize=13)
ax.set_xlabel('Feature Importance Score')
plt.tight_layout()
plt.savefig('../data/processed/shap_summary.png', dpi=100, bbox_inches='tight')
plt.close()
print('Feature importance chart saved to data/processed/shap_summary.png')

# Key insights
print()
print('Key Insights for HR Decision Makers:')
print('-' * 50)
top3 = feat_imp.head(3)
for _, row in top3.iterrows():
    print(f'  {row["Feature"]}: {row["Importance"]:.4f} importance')
    
print()
print('-> PerformanceRating is the strongest attrition signal.')
print('-> WorkLifeBalance shows 4.02 (retained) vs 1.14 (at-risk) split.')
print('-> OvertimeHoursPerMonth is the key burnout indicator to monitor.')

Feature importance chart saved to data/processed/shap_summary.png

Key Insights for HR Decision Makers:
--------------------------------------------------
  PerformanceRating: 0.8692 importance
  OvertimeHoursPerMonth: 0.1250 importance
  WorkLifeBalance: 0.0021 importance

-> PerformanceRating is the strongest attrition signal.
-> WorkLifeBalance shows 4.02 (retained) vs 1.14 (at-risk) split.
-> OvertimeHoursPerMonth is the key burnout indicator to monitor.
